# Cosmos SAE Training and Feature Browser

Run this notebook on the RunPod checkout. It assumes the repo is at `/workspace/cosmos-sae-reasoner` and that the Cosmos3-Nano Hugging Face cache is under `/workspace/.cache/huggingface`.

The default cells reuse the small activation smoke set. To run a larger experiment, change `ACTIVATION_DIR`, `MAX_EXAMPLES`, `TRAIN_STEPS`, and output names in the config cell.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time

ROOT = Path('/workspace/cosmos-sae-reasoner')
MODEL_ID = 'nvidia/Cosmos3-Nano'
LAYER = 18

# Existing tiny smoke set with multimodal token maps. Change this for real runs.
MANIFEST = ROOT / 'outputs/sae_reasoner/sample_manifest.jsonl'
ACTIVATION_DIR = ROOT / 'outputs/sae_reasoner/activations/sample_l18_tokenmap'
SAE_OUT = ROOT / 'outputs/sae_reasoner/saes/l18_notebook.pt'
FEATURES_OUT = ROOT / 'outputs/sae_reasoner/reports/l18_notebook_features.jsonl'
REPORT_OUT = ROOT / 'outputs/sae_reasoner/reports/l18_notebook_features.html'

MAX_EXAMPLES = 8
TRAIN_STEPS = 500
BATCH_SIZE = 1024
EXPANSION_FACTOR = 16
TOP_K = 32
RECON_LOSS = 'mse'
FEATURE_L1_COEFF = 0.0
FEATURE_IDS = ''  # empty means first 128 features
TOP_N = 20

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'
token_path = Path('/root/.cache/huggingface/token')
if token_path.exists():
    token = token_path.read_text(encoding='utf-8').strip()
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

def run(cmd: str) -> None:
    print(f'$ {cmd}', flush=True)
    start = time.time()
    subprocess.run(cmd, cwd=ROOT, shell=True, check=True)
    print(f'done in {time.time() - start:.1f}s', flush=True)

print('repo', ROOT)
print('activation_dir', ACTIVATION_DIR)
print('sae_out', SAE_OUT)
print('features_out', FEATURES_OUT)
print('report_out', REPORT_OUT)

In [ ]:
# Quick environment check.
run('git rev-parse --short HEAD')
run('python -m tools.sae_reasoner inspect-model --model-id nvidia/Cosmos3-Nano --init-mode meta')

In [ ]:
# Optional: collect activations. Leave RUN_COLLECTION=False if you already have ACTIVATION_DIR.
RUN_COLLECTION = False

if RUN_COLLECTION:
    run(
        'python -m tools.sae_reasoner collect-activations '
        f'--model-id {MODEL_ID} '
        f'--manifest {MANIFEST} '
        f'--layer {LAYER} '
        f'--output-dir {ACTIVATION_DIR} '
        f'--max-examples {MAX_EXAMPLES}'
    )
else:
    print('Skipping collection. Using existing activation directory.')

print('activation shards:', sorted(p.name for p in ACTIVATION_DIR.glob('*.pt'))[:10])
print('metadata exists:', (ACTIVATION_DIR / 'metadata.jsonl').exists())

In [ ]:
# Inspect activation metadata and token-map coverage.
metadata_path = ACTIVATION_DIR / 'metadata.jsonl'
rows = [json.loads(line) for line in metadata_path.read_text(encoding='utf-8').splitlines()] if metadata_path.exists() else []
for row in rows[:10]:
    print({
        'id': row.get('id'),
        'media_type': row.get('media_type'),
        'num_tokens': row.get('num_tokens'),
        'token_kind_counts': row.get('token_kind_counts'),
        'visual_grid': row.get('visual_grid'),
    })

In [ ]:
# Train the SAE.
run(
    'python -m tools.sae_reasoner train-sae '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--output {SAE_OUT} '
    f'--expansion-factor {EXPANSION_FACTOR} '
    f'--top-k {TOP_K} '
    f'--recon-loss {RECON_LOSS} '
    f'--feature-l1-coeff {FEATURE_L1_COEFF} '
    f'--steps {TRAIN_STEPS} '
    f'--batch-size {BATCH_SIZE}'
)

In [ ]:
# Inspect training metrics.
metrics_path = SAE_OUT.with_suffix('.metrics.jsonl')
metrics = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines()]
print('num metric rows:', len(metrics))
for row in metrics[-10:]:
    print(row)

In [ ]:
# Find top activating examples and render the feature browser.
run(
    'python -m tools.sae_reasoner find-features '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--sae {SAE_OUT} '
    f'--feature-ids "{FEATURE_IDS}" '
    f'--top-n {TOP_N} '
    f'--output {FEATURES_OUT}'
)
run(
    'python -m tools.sae_reasoner render-feature-report '
    f'--features {FEATURES_OUT} '
    f'--output {REPORT_OUT}'
)
print('report:', REPORT_OUT)

In [ ]:
# Preview feature rows with token metadata.
feature_rows = [json.loads(line) for line in FEATURES_OUT.read_text(encoding='utf-8').splitlines()]
print('num feature rows:', len(feature_rows))
for row in feature_rows[:20]:
    token = row.get('token_info') or {}
    print({
        'feature_id': row.get('feature_id'),
        'activation': round(float(row.get('activation', 0.0)), 4),
        'record_id': row.get('record_id'),
        'token_index': row.get('token_index'),
        'kind': token.get('kind'),
        'token_text': token.get('token_text'),
        'visual_position': token.get('visual_position'),
    })

In [ ]:
# Open the rendered report inside Jupyter.
from IPython.display import IFrame, display

relative_report = REPORT_OUT.relative_to(ROOT)
display(IFrame(src=str(relative_report), width='100%', height=900))